# **Qué es el análisis de sentimiento**

Es una tarea de clasificación de texto. Dado un texto, el modelo decide si la intención utilizada es positiva, negativa o neutral.

No es "**entender**" el texto como lo haría una persona. Es detectar patrones de palabras y contexto que, estadísticamente, se asocian a cada categoría.

Para esta práctica en español, el modelo más adecuado es **RoBERTuito**. Es un modelo entrenado específicamente con **500 millones de tweets en español**, lo que le permite entender jerga, emojis, abreviaturas y texto informal.

Está diseñado para redes sociales, que es donde el análisis de sentimiento tiene más aplicación real. Y funciona bien en tareas de sentimiento, odio e ironía en español.

RoBERTuito no se entrenó desde cero. Es un modelo preentrenado: alguien invirtió muchísimo tiempo y recursos en entrenarlo con textos masivos para que "entienda" el español.

Luego ese modelo base se ajustó (mediante fine-tuning) con un conjunto más pequeño de tweets etiquetados como positivos, negativos o neutros. El resultado es un modelo especializado en análisis de sentimiento en español, listo para usar.

Esto es importante porque explica por qué tú no entrenas nada. Usas un modelo que ya existe, ya está entrenado, y solo lo llamas desde Colab. Es exactamente lo que haría un desarrollador profesional: **no reinventar la rueda, usar lo que ya funciona**.

# **Hugging Face**

Hugging Face es la plataforma donde viven estos modelos. Su función **pipeline** es una abstracción: tú le dices "quiero realizar un análisis de sentimiento" y "usa este modelo", y ella se encarga de todo lo demás (preprocesar el texto, pasarlo por el modelo, devolver el resultado).

Solo se necesita entender que pipeline es el "botón" que conecta el texto con un modelo ya entrenado.

El modelo devuelve dos cosas:

    Una etiqueta: POS, NEG o NEU

    Una confianza (score), un número entre 0 y 1

La confianza no es lo mismo que **"acierto"**. Es la seguridad del modelo en su propia predicción. Un score de 0.99 no garantiza que tenga razón; un score de 0.51 en una frase ambigua es normal y no es un fallo.

El modelo no lee intenciones, lee patrones. El guiño o el signo de interrogación alteran el contexto que el modelo aprendió durante el preentrenamiento, y eso mueve la predicción.

In [2]:
from transformers import pipeline

clasificador = pipeline("sentiment-analysis", model="pysentimiento/robertuito-sentiment-analysis")
# carga un modelo de inteligencia artificial
# entrenado para analizar el sentimiento
# de textos en español, con especial enfoque
# en lenguaje informal y redes sociales

# === este es el texto  que se clasifica ===
texto = "Me encanta este módulo de IA"
# =====================================

resultado = clasificador(texto)
print(resultado)

config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  435MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

[{'label': 'POS', 'score': 0.9514580368995667}]


In [ ]:
resultado = clasificador("Esto es horrible.")
print(resultado)

[{'label': 'NEG', 'score': 0.9667719602584839}]


In [6]:
# clasifica un texto que sea negativo

texto = "El servicio de atención al cliente fue pésimo, nunca me resolvieron el problema."
resultado = clasificador(texto)
print(resultado)

[{'label': 'NEG', 'score': 0.962997317314148}]


In [5]:
# clasifica un texto neutro
texto = "El evento se llevará a cabo el próximo martes a las 18:00 horas en el auditorio principal."
resultado = clasificador(texto)
print(resultado)

[{'label': 'NEU', 'score': 0.7417901158332825}]


In [4]:
# clasifica un texto irónico
texto = "¡Qué maravilla! Justo lo que necesitaba, que se me rompa el coche el primer día de vacaciones"
resultado = clasificador(texto)
print(resultado)

[{'label': 'NEG', 'score': 0.7296634316444397}]


In [16]:
# clasifica un texto usando emojis
texto = "Esta nueva actualización de la app está de locos 🤯 Me encanta 😍"
resultado = clasificador(texto)
print(resultado)

[{'label': 'POS', 'score': 0.9738227128982544}]


**¿Cómo funciona el análisis de sentimientos? ¿Qué límites tiene?**

El modelo no entiende el texto de la misma manera que lo haría un ser humano. Funciona detectando patrones de palabras y contextos (incluyendo puntuación, abreviaturas y emojis) que estadísticamente se han asociado a una etiqueta específica (positivo, negativo o neutro) durante su fase de entrenamiento. Todo esto ocurre a través de una abstracción llamada pipeline, que procesa el texto introducido y lo pasa por el modelo preentrenado para generar una clasificación.

Los límites que tienes son:

1- La puntuación de confianza no es infalible: El score indica qué tan seguro está el modelo de su predicción, pero una alta seguridad no garantiza que la clasificación sea un acierto absoluto en el mundo real.   

2- Sensibilidad al contexto: Modificaciones sutiles, como el uso de un guiño o signos de interrogación, pueden alterar por completo los patrones que el modelo identifica, provocando saltos en la predicción.   

3- Falta de comprensión profunda: Al ser un análisis estadístico de patrones y no de comprensión lectora real, puede tener dificultades con sarcasmos extremadamente sutiles, dobles sentidos muy locales o lenguaje técnico no incluido en los 500 millones de tweets con los que fue entrenado originalmente.   


# **Práctica**

Implementa en python un analizador de comentarios. Los comentarios positivos y neutros serán aceptados, salvo que contengan alguna palabra de la lista negra de palabras. Los comentarios negativos que contengan alguna palabra de la lista negra o que tengan un score de más de 0.5 serán rechazadas.

In [15]:
def analizador_de_comentarios(comentario, lista_negra):

    palabras_comentario = comentario.lower().split()

    for palabra_prohibida in lista_negra:
        if palabra_prohibida.lower() in palabras_comentario:
            return f"RECHAZADO: Contiene palabra prohibida ('{palabra_prohibida}')."

    resultado = clasificador(comentario)[0]
    etiqueta = resultado['label']
    score = resultado['score']

    if etiqueta == 'NEG':
        if score > 0.5:
            return f"RECHAZADO: Comentario muy negativo (Score: {score:.2f})."
        else:

            return f"ACEPTADO: Comentario negativo pero con baja confianza (Score: {score:.2f})."

    return f"ACEPTADO: Comentario {etiqueta} (Score: {score:.2f})."

print("TEST DEL ANALIZADOR\n")
palabras_bloqueadas = ["estafa", "basura", "fraude", "odio"]

comentarios_prueba = [
    "Este curso es una maravilla, aprendí mucho.",
    "El envío se realizó el martes por la mañana.",
    "El soporte técnico es una basura, nadie contesta.",
    "No me convenció mucho la interfaz, es algo confusa.",
    "Es una estafa total aunque me guste un poco."
]

for texto_prueba in comentarios_prueba:
    print(f"Comentario: '{texto_prueba}'")
    print(analizador_de_comentarios(texto_prueba, palabras_bloqueadas))
    print("-------------------------------------------------------------")


TEST DEL ANALIZADOR

Comentario: 'Este curso es una maravilla, aprendí mucho.'
ACEPTADO: Comentario POS (Score: 0.97).
-------------------------------------------------------------
Comentario: 'El envío se realizó el martes por la mañana.'
ACEPTADO: Comentario NEU (Score: 0.91).
-------------------------------------------------------------
Comentario: 'El soporte técnico es una basura, nadie contesta.'
RECHAZADO: Comentario muy negativo (Score: 0.97).
-------------------------------------------------------------
Comentario: 'No me convenció mucho la interfaz, es algo confusa.'
RECHAZADO: Comentario muy negativo (Score: 0.86).
-------------------------------------------------------------
Comentario: 'Es una estafa total aunque me guste un poco.'
RECHAZADO: Contiene palabra prohibida ('estafa').
-------------------------------------------------------------
